# Payment Reconciliation: Ledger vs. Gateway Export

Reusable `reconcile_payments(ledger_df, gateway_df)` function that compares an
internal ledger against a payment gateway export and returns four discrepancy
DataFrames:

1. **Missing in gateway** — transactions in the ledger but absent from the gateway export
2. **Missing in ledger** — transactions in the gateway export but absent from the ledger (extras)
3. **Amount mismatches** — transactions in both, with differing `amount_inr` (includes computed `amount_diff`)
4. **Status mismatches** — transactions in both, with differing `status`

Set operations on `transaction_id` are used for (1) and (2); `pd.merge` is used for the pairwise comparisons in (3) and (4).

In [2]:
import pandas as pd

## `reconcile_payments` function

In [3]:
def reconcile_payments(ledger_df: pd.DataFrame, gateway_df: pd.DataFrame):

    ledger_ids = set(ledger_df['transaction_id'])
    gateway_ids = set(gateway_df['transaction_id'])

    # --- Set operations on transaction_id -----------------------------------
    missing_in_gateway_ids = ledger_ids - gateway_ids
    missing_in_ledger_ids = gateway_ids - ledger_ids

    missing_in_gateway = (
        ledger_df[ledger_df['transaction_id'].isin(missing_in_gateway_ids)]
        .sort_values('transaction_id')
        .reset_index(drop=True)
    )
    missing_in_ledger = (
        gateway_df[gateway_df['transaction_id'].isin(missing_in_ledger_ids)]
        .sort_values('transaction_id')
        .reset_index(drop=True)
    )

    # --- Pairwise comparison via merge on shared transaction_ids ------------
    common = pd.merge(
        ledger_df,
        gateway_df,
        on='transaction_id',
        how='inner',
        suffixes=('_ledger', '_gateway'),
    )

    # Amount mismatches
    amount_mismatches = common[
        common['amount_inr_ledger'] != common['amount_inr_gateway']
    ].copy()
    amount_mismatches['amount_diff'] = (
        amount_mismatches['amount_inr_gateway'] - amount_mismatches['amount_inr_ledger']
    )
    amount_mismatches = amount_mismatches[
        ['transaction_id', 'amount_inr_ledger', 'amount_inr_gateway', 'amount_diff']
    ].sort_values('transaction_id').reset_index(drop=True)

    # Status mismatches
    status_mismatches = common[
        common['status_ledger'] != common['status_gateway']
    ][['transaction_id', 'status_ledger', 'status_gateway']].sort_values(
        'transaction_id'
    ).reset_index(drop=True)

    return missing_in_gateway, missing_in_ledger, amount_mismatches, status_mismatches

## Load the data

In [4]:
ledger = pd.read_csv('/content/ledger.csv')
gateway = pd.read_csv('/content/gateway_export.csv')

n = len(ledger)
print(f'Ledger rows:  {len(ledger)}')
print(f'Gateway rows: {len(gateway)}')

Ledger rows:  547
Gateway rows: 530


## Run the reconciliation

In [5]:
missing_in_gateway, missing_in_ledger, amount_mismatches, status_mismatches = \
    reconcile_payments(ledger, gateway)

## Discrepancy counts

Expected injection rates from `generate_data.py`: ~5% missing in gateway, ~3% amount
mismatches, ~2% extra in gateway (missing in ledger), ~2% status mismatches.

In [6]:
print(f'1) Missing in gateway (present in ledger, absent from gateway): '
      f'{len(missing_in_gateway)}  ({len(missing_in_gateway) / n:.1%} of ledger)')
print(f'2) Missing in ledger (extra in gateway):                        '
      f'{len(missing_in_ledger)}  ({len(missing_in_ledger) / n:.1%} of ledger)')
print(f'3) Amount mismatches:                                           '
      f'{len(amount_mismatches)}  ({len(amount_mismatches) / n:.1%} of ledger)')
print(f'4) Status mismatches:                                           '
      f'{len(status_mismatches)}  ({len(status_mismatches) / n:.1%} of ledger)')

1) Missing in gateway (present in ledger, absent from gateway): 27  (4.9% of ledger)
2) Missing in ledger (extra in gateway):                        10  (1.8% of ledger)
3) Amount mismatches:                                           16  (2.9% of ledger)
4) Status mismatches:                                           9  (1.6% of ledger)


## 1) Missing in gateway

In [7]:
print(f'{len(missing_in_gateway)} rows')
missing_in_gateway.head(10)

27 rows


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
0,TXN100000,350,16,2026-01-29 11:13:00,2999,Wallet,captured,85
1,TXN100005,196,3,2026-01-06 22:06:00,99,UPI,captured,60
2,TXN100036,321,16,2026-01-25 16:31:00,49,UPI,captured,35
3,TXN100044,309,15,2026-01-24 23:34:00,99,Netbanking,captured,62
4,TXN100056,132,24,2026-01-07 14:44:00,299,Wallet,captured,21
5,TXN100074,244,32,2026-01-18 07:44:00,149,UPI,captured,57
6,TXN100092,157,37,2026-01-28 18:27:00,499,Wallet,captured,79
7,TXN100130,211,3,2026-01-13 07:54:00,149,UPI,chargeback,40
8,TXN100134,74,27,2026-01-17 14:00:00,799,UPI,captured,83
9,TXN100188,32,31,2026-01-15 20:01:00,299,UPI,captured,51


## 2) Missing in ledger / extra in gateway

In [8]:
print(f'{len(missing_in_ledger)} rows')
missing_in_ledger.head(10)

10 rows


,transaction_id,user_id,merchant_id,transaction_time,amount_inr,payment_method,status,risk_score
0,TXNX9000,262,25,2026-01-14 00:00:00,149,Netbanking,captured,54
1,TXNX9001,96,32,2026-01-21 00:00:00,4999,Wallet,captured,71
2,TXNX9002,86,32,2026-01-10 00:00:00,149,Wallet,captured,40
3,TXNX9003,231,40,2026-01-02 00:00:00,799,UPI,captured,62
4,TXNX9004,70,13,2026-01-27 00:00:00,1499,Netbanking,captured,52
5,TXNX9005,252,27,2026-01-23 00:00:00,2999,Netbanking,captured,21
6,TXNX9006,43,37,2026-01-01 00:00:00,299,Card,captured,4
7,TXNX9007,141,15,2026-01-18 00:00:00,499,Wallet,captured,99
8,TXNX9008,235,37,2026-01-24 00:00:00,2999,UPI,captured,73
9,TXNX9009,59,18,2026-01-25 00:00:00,4999,Card,captured,69


## 3) Amount mismatches

In [9]:
print(f'{len(amount_mismatches)} rows')
amount_mismatches.head(10)

16 rows


,transaction_id,amount_inr_ledger,amount_inr_gateway,amount_diff
0,TXN100011,2999,2949,-50
1,TXN100019,49,99,50
2,TXN100039,1499,1399,-100
3,TXN100204,49,-51,-100
4,TXN100218,49,-1,-50
5,TXN100252,299,349,50
6,TXN100270,799,899,100
7,TXN100278,149,99,-50
8,TXN100284,1499,1399,-100
9,TXN100286,49,149,100


## 4) Status mismatches

In [10]:
print(f'{len(status_mismatches)} rows')
status_mismatches.head(10)

9 rows


,transaction_id,status_ledger,status_gateway
0,TXN100042,captured,failed
1,TXN100215,captured,failed
2,TXN100267,captured,failed
3,TXN100300,captured,failed
4,TXN100392,captured,failed
5,TXN100414,captured,failed
6,TXN100487,captured,failed
7,TXN200008,chargeback,failed
8,TXN300027,captured,failed
